# LLM Inference Benchmark Suite — Colab Orchestrator

Compares FP16, GPTQ, AWQ, GGUF, and TensorRT-LLM inference on a single
open-weight model, using **one isolated virtual environment per technique**
(see `README.md` and `docs/environment_notes.md` for why).

**Recommended runtime: A100 (40GB).** Phases 1-4 also work on a T4;
Phase 5 (TensorRT-LLM) requires Ampere or newer and will fail fast with a
clear assertion error otherwise.


In [ ]:
# Phase 0 -- Clone repo and confirm GPU
!git clone -q https://github.com/arkanathroy/llm-inference-benchmark-suite.git 2>/dev/null || echo "repo already present"
%cd llm-inference-benchmark-suite

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv


## Phase 1 -- Build the five isolated environments

In [ ]:
# Phase 1.1 -- FP16 baseline env
!bash envs/fp16/setup.sh


In [ ]:
# Phase 1.2 -- GPTQ env (GPTQModel)
!bash envs/gptq/setup.sh


In [ ]:
# Phase 1.3 -- AWQ env (AutoAWQ)
!bash envs/awq/setup.sh


In [ ]:
# Phase 1.4 -- GGUF env (llama-cpp-python + llama.cpp CLI tools)
!bash envs/gguf/setup.sh


In [ ]:
# Phase 1.5 -- TensorRT-LLM env (A100/H100 only)
import torch
major, minor = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
if major >= 8:
    !bash envs/trtllm/setup.sh
else:
    print(f"Skipping TensorRT-LLM env: detected compute capability sm_{major}{minor}, "
          f"need sm_80 or higher (A100/H100). Switch runtime to continue with Phase 5.")


## Phase 2 -- Quantize the model with each technique

In [ ]:
# Phase 2.1 -- GPTQ quantization (runs inside envs/gptq/venv)
import sys
sys.path.insert(0, "src")
from env_runner import run_in_env

run_in_env("gptq", "src/quantize_gptq.py", [])


In [ ]:
# Phase 2.2 -- AWQ quantization (runs inside envs/awq/venv)
run_in_env("awq", "src/quantize_awq.py", [])


In [ ]:
# Phase 2.3 -- GGUF conversion + quantization (runs inside envs/gguf/venv)
run_in_env("gguf", "src/convert_gguf.py", [])


In [ ]:
# Phase 2.4 -- TensorRT-LLM engine build (Ampere+/A100 only; runs inside envs/trtllm/venv)
from pathlib import Path
if Path("envs/trtllm/venv").exists():
    run_in_env("trtllm", "src/build_trtllm_engine.py", [])
else:
    print("Skipping: trtllm env was not built in Phase 1.5 (non-Ampere GPU).")


## Phase 3 -- Benchmark FP16 baseline

Starts a vLLM server inside `envs/fp16/venv`, fires load against it via
`benchmark_runner.py`, then shuts the server down.


In [ ]:
# Phase 3.1 -- FP16 baseline benchmark
import subprocess, time, sys
sys.path.insert(0, "src")
from config import CONFIG

model_id = CONFIG.model.hf_repo
port = CONFIG.server.vllm_port

server_proc = subprocess.Popen([
    "envs/fp16/venv/bin/python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", model_id,
    "--host", CONFIG.server.host,
    "--port", str(port),
    "--gpu-memory-utilization", str(CONFIG.server.gpu_memory_utilization),
    "--max-model-len", str(CONFIG.model.max_model_len),
])

import httpx
base_url = f"http://{CONFIG.server.host}:{port}"
for _ in range(60):
    try:
        if httpx.get(f"{base_url}/health", timeout=2).status_code == 200:
            break
    except Exception:
        pass
    time.sleep(2)

run_in_env(
    "fp16", "src/benchmark_runner.py",
    ["--technique", "fp16",
     "--base_url", f"{base_url}/v1",
     "--model_id", model_id],
)

server_proc.terminate()
server_proc.wait(timeout=15)


## Phase 4 -- Benchmark GPTQ

In [ ]:
# Phase 4.1 -- GPTQ benchmark
gptq_model_dir = CONFIG.gptq.output_dir

server_proc = subprocess.Popen([
    "envs/gptq/venv/bin/python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", gptq_model_dir,
    "--host", CONFIG.server.host,
    "--port", str(port),
    "--gpu-memory-utilization", str(CONFIG.server.gpu_memory_utilization),
    "--max-model-len", str(CONFIG.model.max_model_len),
    "--quantization", "gptq",
])

for _ in range(60):
    try:
        if httpx.get(f"{base_url}/health", timeout=2).status_code == 200:
            break
    except Exception:
        pass
    time.sleep(2)

run_in_env(
    "gptq", "src/benchmark_runner.py",
    ["--technique", "gptq",
     "--base_url", f"{base_url}/v1",
     "--model_id", gptq_model_dir],
)

server_proc.terminate()
server_proc.wait(timeout=15)


## Phase 5 -- Benchmark AWQ

In [ ]:
# Phase 5.1 -- AWQ benchmark
awq_model_dir = CONFIG.awq.output_dir

server_proc = subprocess.Popen([
    "envs/awq/venv/bin/python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", awq_model_dir,
    "--host", CONFIG.server.host,
    "--port", str(port),
    "--gpu-memory-utilization", str(CONFIG.server.gpu_memory_utilization),
    "--max-model-len", str(CONFIG.model.max_model_len),
    "--quantization", "awq",
])

for _ in range(60):
    try:
        if httpx.get(f"{base_url}/health", timeout=2).status_code == 200:
            break
    except Exception:
        pass
    time.sleep(2)

run_in_env(
    "awq", "src/benchmark_runner.py",
    ["--technique", "awq",
     "--base_url", f"{base_url}/v1",
     "--model_id", awq_model_dir],
)

server_proc.terminate()
server_proc.wait(timeout=15)


## Phase 6 -- Benchmark GGUF

Uses llama.cpp's own server binary instead of vLLM (vLLM's GGUF support is
explicitly experimental upstream and only loads single-file checkpoints).


In [ ]:
# Phase 6.1 -- GGUF benchmark (Q4_K_M variant)
gguf_dir = CONFIG.gguf.output_dir
gguf_file = f"{gguf_dir}/model-Q4_K_M.gguf"
llamacpp_port = CONFIG.server.llamacpp_port

server_proc = subprocess.Popen([
    "envs/gguf/llama.cpp/build/bin/llama-server",
    "-m", gguf_file,
    "--host", CONFIG.server.host,
    "--port", str(llamacpp_port),
    "-ngl", "-1",
    "-c", str(CONFIG.model.max_model_len),
])

llamacpp_base_url = f"http://{CONFIG.server.host}:{llamacpp_port}"
for _ in range(60):
    try:
        if httpx.get(f"{llamacpp_base_url}/health", timeout=2).status_code == 200:
            break
    except Exception:
        pass
    time.sleep(2)

run_in_env(
    "gguf", "src/benchmark_runner.py",
    ["--technique", "gguf_q4_k_m",
     "--base_url", f"{llamacpp_base_url}/v1",
     "--model_id", gguf_file],
)

server_proc.terminate()
server_proc.wait(timeout=15)


## Phase 7 -- Benchmark TensorRT-LLM (A100/H100 only)

TensorRT-LLM engines run through their own native runtime
(`tensorrt_llm.runtime.ModelRunner`), not an HTTP server, so this phase
calls `trtllm_bench.py` directly rather than reusing `benchmark_runner.py`.
This is the concrete replacement for the old notebook's Phase 9.3
`for ... : pass` placeholder loop.


In [ ]:
# Phase 7.1 -- TensorRT-LLM native benchmark
if Path("envs/trtllm/venv").exists():
    engine_dir = f"{CONFIG.trtllm.output_dir}/trt_engine"
    run_in_env(
        "trtllm", "src/trtllm_bench.py",
        ["--engine_dir", engine_dir,
         "--output", "results/trtllm_benchmark.json"],
    )
else:
    print("Skipping: no trtllm engine was built (non-Ampere GPU in Phase 1.5/2.4).")


## Phase 8 -- Accuracy check (lm-eval) across techniques

Runs `lm-eval` against each served endpoint to measure task accuracy
delta introduced by each quantization technique, relative to the FP16
baseline. Skipped for TensorRT-LLM in this default config since lm-eval's
API-based runner targets HTTP servers, not native TRT engines.


In [ ]:
# Phase 8.1 -- Accuracy comparison (illustrative; expand per technique as needed)
# Each technique's venv already has lm-eval installed (see envs/*/setup.sh).
# Example invocation pattern for the FP16 baseline server:
#
# run_in_env(
#     "fp16", "-m",
#     ["lm_eval", "--model", "local-completions",
#      "--model_args", f"base_url={base_url}/v1/completions,model={model_id}",
#      "--tasks", ",".join(CONFIG.eval.tasks),
#      "--limit", str(CONFIG.eval.limit)],
# )
#
# Repeat with each technique's own base_url/model_id while its server is running.
print("See commented template above -- run per-technique with its server active.")


## Phase 9 -- Aggregate results and plot comparison charts

In [ ]:
# Phase 9.1 -- Load combined results CSV
import pandas as pd

results_csv = Path(CONFIG.output.results_dir) / CONFIG.output.csv_name
df = pd.read_csv(results_csv) if results_csv.exists() else pd.DataFrame()
df


In [ ]:
# Phase 9.2 -- Fold in TensorRT-LLM results (separate JSON schema) if present
import json

trtllm_json = Path("results/trtllm_benchmark.json")
if trtllm_json.exists():
    trt_results = json.load(open(trtllm_json))
    trt_df = pd.DataFrame(trt_results).rename(columns={
        "avg_latency_s": "avg_latency_s",
        "tokens_per_second": "avg_tokens_per_second",
    })
    df = pd.concat([df, trt_df], ignore_index=True, sort=False)

df


In [ ]:
# Phase 9.3 -- Plot tokens/sec by technique and batch size
import plotly.express as px

if not df.empty:
    fig = px.bar(
        df, x="batch_size", y="avg_tokens_per_second", color="technique",
        barmode="group", title="Throughput by technique and batch size",
        labels={"avg_tokens_per_second": "tokens / second", "batch_size": "batch size"},
    )
    fig.write_image("results/benchmark_comparison.png", scale=2)
    fig.show()
else:
    print("No results yet -- run Phases 3-7 first.")
